# WriterAgent 動作確認

## 概要

WriterAgent は記事生成エージェントです。与えられたトピック・キーワード・目標文字数・トーンに基づき、
LangGraph ワークフローで高品質な記事を自動生成します。

### ワークフロー

1. **角度提案** (AngleProposalNode) — トピックに対する記事の切り口を複数提案
2. **角度選択** (AngleSelectionNode) — 最適な切り口を自動選択
3. **構成計画** (PlannerNode) — 記事のセクション構成を計画
4. **セクション執筆** (ExecutorNode) — 各セクションを執筆
5. **品質チェック** (ReflectorNode) — 記事の品質を評価し、必要に応じて再執筆
6. **統合** (IntegratorNode) — セクションを統合して最終記事を生成

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# tools/ ディレクトリを特定
notebook_dir = Path.cwd()
for candidate in [
    notebook_dir.parents[3],  # notebook → writer → agents → src → tools
    notebook_dir.parents[4],
    notebook_dir,
]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        tools_dir = str(candidate)
        break
else:
    tools_dir = str(notebook_dir)

if tools_dir not in sys.path:
    sys.path.insert(0, tools_dir)

print(f"Python: {sys.executable}")
print(f"tools_dir: {tools_dir}")

# python-dotenv がなければインストール
try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv"])
    from dotenv import load_dotenv

env_path = Path(tools_dir) / ".env"
load_dotenv(env_path)
print(f".env loaded from: {env_path}")
print(f"OPENAI_API_KEY set: {'OPENAI_API_KEY' in os.environ}")

## 入力データの説明

`WriterInput` は以下のフィールドを持ちます:

| フィールド | 型 | デフォルト | 説明 |
|---|---|---|---|
| `topic` | `str` | (必須) | 記事のトピック |
| `keywords` | `list[str]` | (必須) | 使用するキーワード（1つ以上） |
| `target_length` | `int` | `2000` | 目標文字数 |
| `tone` | `str` | `"informative"` | 記事のトーン |

In [ ]:
from src.agents.writer.schemas import WriterInput

input_data = WriterInput(
    topic="新NISAの始め方",
    keywords=["新NISA", "投資信託", "つみたて投資枠"],
    target_length=2000,
    tone="informative",
)

print(f"topic: {input_data.topic}")
print(f"keywords: {input_data.keywords}")
print(f"target_length: {input_data.target_length}")
print(f"tone: {input_data.tone}")

## エージェント実行

`WriterAgent` をインスタンス化し、`run()` メソッドで記事を生成します。

> **注意**: 実行すると OpenAI API が呼び出されます。`OPENAI_API_KEY` が設定されていることを確認してください。

In [ ]:
from src.agents.writer.agent import WriterAgent

agent = WriterAgent()
result = agent.run(input_data)

print(f"生成完了: {result.title}")

## 結果の確認

`WriterOutput` は以下のフィールドを持ちます:

| フィールド | 型 | 説明 |
|---|---|---|
| `title` | `str` | 記事タイトル |
| `description` | `str` | メタディスクリプション |
| `content` | `str` | 記事本文（Markdown） |
| `keywords_used` | `list[str]` | 使用されたキーワード |
| `sections` | `list[Section]` | セクション情報 |
| `summary` | `str` | 執筆結果のサマリー |

In [ ]:
from IPython.display import Markdown, display

print("=" * 60)
print(f"タイトル: {result.title}")
print(f"概要: {result.description}")
print(f"使用キーワード: {result.keywords_used}")
print(f"セクション数: {len(result.sections)}")
print(f"サマリー: {result.summary}")
print("=" * 60)

# セクション一覧
print("\n--- セクション一覧 ---")
for i, section in enumerate(result.sections, 1):
    print(f"  {i}. [H{section.level}] {section.heading}")

# 記事本文をMarkdownとして表示
print("\n--- 記事本文 ---\n")
display(Markdown(result.content))